# Treinar um LLM do zero, em 5 minutos

O treino inteiro de um modelo de linguagem, do texto cru ao modelo
escrevendo, em Keras. É o mesmo modelo das Aulas 10 e 11 (809.728 pesos,
treinado em Machado de Assis) e o mesmo que roda no
[Explorador do LLM](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/explorador),
só que agora na placa de vídeo do Colab.

**Antes de rodar:** menu *Ambiente de execução → Alterar o tipo de
ambiente de execução → T4 GPU*. Depois, *Ambiente de execução → Executar
tudo*. Leva uns 7 minutos: 5 de treino e o resto de preparação.

Não há exercício aqui. Leia o código de cima para baixo: cada célula é
uma etapa do treino.

## 1. O ambiente

Confere se há placa de vídeo. Com ela, ligamos duas otimizações:

- **precisão mista**: as contas pesadas em 16 bits (duas vezes mais
  rápidas na T4), e os pesos guardados em 32 bits;
- **XLA**: o TensorFlow compila o passo de treino inteiro numa função só,
  em vez de chamar cada operação separadamente.

In [ ]:
import json
import math
import re
import time
import urllib.request
import warnings

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras import ops

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    keras.mixed_precision.set_global_policy("mixed_float16")
    print("Placa de vídeo:", tf.config.experimental.get_device_details(gpus[0]).get("device_name", "GPU"))
else:
    print("Sem placa de vídeo: o treino vai rodar no processador, bem mais devagar.")
keras.utils.set_random_seed(42)
warnings.filterwarnings("ignore", message=".*shuffle.*")


def br(n):                                       # 1484827 -> "1.484.827"
    return f"{n:,}".replace(",", ".")

## 2. O texto

A obra de Machado de Assis em domínio público, num arquivo só:
3,7 milhões de caracteres.

In [ ]:
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/"
texto = urllib.request.urlopen(URL_DADOS + "machado.txt").read().decode("utf-8")
print(br(len(texto)), "caracteres")
print(texto[100_000:100_400])

## 3. O tokenizador

O BPE da Aula 9, com as 768 fusões já aprendidas: o texto é separado em
palavras, cada palavra vira bytes, e os pares de bytes se fundem na
ordem em que o BPE os aprendeu. Vocabulário: 256 bytes + 768 fusões =
1.024 tokens.

In [ ]:
bpe = json.loads(urllib.request.urlopen(URL_DADOS + "machado_bpe.json").read())
padrao = re.compile(bpe["padrao"])
ordem_da_fusao = {tuple(par): k for k, par in enumerate(bpe["fusoes"])}
pecas = [bytes([i]) for i in range(256)]
for a, b in bpe["fusoes"]:
    pecas.append(pecas[a] + pecas[b])


def codificar_palavra(palavra):
    ids = list(palavra.encode("utf-8"))
    while len(ids) >= 2:
        pares = [(ordem_da_fusao.get(par, 10**9), i) for i, par in enumerate(zip(ids, ids[1:]))]
        melhor, i = min(pares)
        if melhor == 10**9:                      # nenhum par conhecido: acabou
            break
        ids[i:i + 2] = [256 + melhor]
    return ids


memoria = {}


def codificar(texto):
    saida = []
    for palavra in padrao.findall(texto):
        if palavra not in memoria:
            memoria[palavra] = codificar_palavra(palavra)
        saida.extend(memoria[palavra])
    return saida


def decodificar(ids):
    return b"".join(pecas[i] for i in ids).decode("utf-8", errors="replace")


print(codificar("A casa de Capitú"), "→", decodificar(codificar("A casa de Capitú")))

O livro inteiro vira uma fila de números. Os primeiros 90% são para
treinar; os últimos 10% ficam guardados para medir se o modelo aprendeu
ou decorou (a validação). Leva uns 30 segundos.

In [ ]:
inicio = time.time()
tokens = np.array(codificar(texto), dtype="int32")
corte = int(0.9 * len(tokens))
treino, validacao = tokens[:corte], tokens[corte:]
print(f"{br(len(tokens))} tokens, {len(texto) / len(tokens):.1f} caracteres por token, "
      f"em {time.time() - inicio:.0f} s")
print(f"treino: {br(len(treino))}   validação: {br(len(validacao))}")

## 4. Os lotes

Um exemplo de treino é uma janela de 128 tokens, e o alvo é a mesma
janela andada uma casa. Um lote junta 128 janelas sorteadas: 16.384
previsões por passo. Os lotes são sorteados dentro do TensorFlow
(`tf.data`), para a placa de vídeo nunca ficar esperando o Python.

In [ ]:
BLOCO = 128          # janela de contexto
LOTE = 128           # janelas por passo (na placa de vídeo cabe bem mais que no processador)


def lotes(dados):
    dados = tf.constant(dados)
    maximo = len(dados) - BLOCO - 1

    def sortear(_):
        inicios = tf.random.uniform([LOTE], 0, maximo, dtype=tf.int32)
        janelas = tf.gather(dados, inicios[:, None] + tf.range(BLOCO + 1)[None, :])
        return janelas[:, :-1], janelas[:, 1:]          # entrada e alvo, andado uma casa

    return tf.data.Dataset.range(10**9).map(sortear, num_parallel_calls=tf.data.AUTOTUNE) \
                                       .prefetch(tf.data.AUTOTUNE)


entrada, alvo = next(iter(lotes(treino)))
print("entrada:", entrada.shape, "  alvo:", alvo.shape)
print("entrada:", repr(decodificar(entrada[0, :12].numpy())))
print("alvo:   ", repr(decodificar(alvo[0, :12].numpy())))

## 5. O modelo

O GPT pequeno da Aula 10, com as camadas prontas do Keras:

1. embedding do token + embedding da posição (128 números por token);
2. quatro blocos: normaliza → atenção com máscara → soma; normaliza →
   camada densa → soma;
3. uma última normalização e a saída, com 1.024 pontuações por posição.

In [ ]:
VOCABULARIO, DIMENSAO, CABECAS, CAMADAS = 1024, 128, 4, 4


class TokenEPosicao(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.token = keras.layers.Embedding(VOCABULARIO, DIMENSAO, name="token")
        self.posicao = keras.layers.Embedding(BLOCO, DIMENSAO, name="posicao")

    def call(self, ids):
        return self.token(ids) + self.posicao(ops.arange(ops.shape(ids)[1]))


def montar_modelo():
    entrada = keras.Input(shape=(None,), dtype="int32")
    x = TokenEPosicao(name="embedding")(entrada)
    for i in range(CAMADAS):
        h = keras.layers.LayerNormalization(name=f"norma_a{i}")(x)
        h = keras.layers.MultiHeadAttention(num_heads=CABECAS, key_dim=DIMENSAO // CABECAS,
                                            name=f"atencao{i}")(h, h, use_causal_mask=True)
        x = x + h                                                   # atalho
        h = keras.layers.LayerNormalization(name=f"norma_b{i}")(x)
        h = keras.layers.Dense(2 * DIMENSAO, activation="gelu", name=f"densa{i}")(h)
        x = x + keras.layers.Dense(DIMENSAO, name=f"volta{i}")(h)    # atalho
    x = keras.layers.LayerNormalization(name="norma_final")(x)
    # A saída fica em 32 bits mesmo com a precisão mista: a softmax precisa de precisão.
    saida = keras.layers.Dense(VOCABULARIO, dtype="float32", name="saida")(x)
    return keras.Model(entrada, saida)


modelo = montar_modelo()
print(br(modelo.count_params()), "pesos")

## 6. A perda e o otimizador

- **Perda**: a entropia cruzada entre as 1.024 pontuações e o token que de
  fato veio, em cada uma das 16.384 posições do lote.
- **Otimizador**: AdamW, com o corte do gradiente (`global_clipnorm`).
- `jit_compile=True` liga o XLA; `steps_per_execution` roda 50 passos
  seguidos na placa de vídeo antes de voltar ao Python.

Antes de treinar, o teste do passo zero: o modelo novo chuta igual entre
1.024 tokens, e a perda tem que dar perto de ln(1024) = 6,93.

In [ ]:
TAXA_MAXIMA = 1e-3

modelo.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=TAXA_MAXIMA, weight_decay=0.1, beta_2=0.95,
                                     global_clipnorm=1.0),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    jit_compile=bool(gpus),
    steps_per_execution=50,
)
perda_zero = modelo.evaluate(lotes(validacao), steps=10, verbose=0)
print(f"passo zero: perda {perda_zero:.2f} (o chute ao acaso dá {math.log(1024):.2f})")

## 7. O relógio, a conferência e o melhor modelo

Três *callbacks*, funções que o `fit` chama no meio do treino:

- o **relógio** ajusta a taxa de aprendizado pela fração do tempo que
  passou (aquecimento no começo, resfriamento até o fim) e para o treino
  quando os 5 minutos acabam;
- a cada **conferência** (a cada 500 passos), ele mede a validação e
  mostra uma frase escrita pelo modelo naquele momento;
- o **`ModelCheckpoint`** salva os pesos sempre que a validação bate o
  recorde.

In [ ]:
MINUTOS_DE_TREINO = 5
PASSOS_POR_CONFERENCIA = 500


def gerar(modelo, comeco, quantos=40, temperatura=0.8):
    ids = codificar(comeco)
    for _ in range(quantos):
        logits = modelo(np.array([ids[-BLOCO:]]), training=False)[0, -1]
        ids.append(int(keras.random.categorical(logits[None] / temperatura, 1)[0, 0]))
    return decodificar(ids)


class Relogio(keras.callbacks.Callback):
    def __init__(self, minutos):
        super().__init__()
        self.minutos = minutos
        self.historico = {"passo": [], "minuto": [], "treino": [], "validacao": []}

    def on_train_begin(self, logs=None):
        self.inicio = time.time()

    def on_train_batch_begin(self, passo, logs=None):
        fracao = min((time.time() - self.inicio) / (self.minutos * 60), 1)
        if fracao >= 1:
            self.model.stop_training = True
        if fracao < 0.03:
            taxa = TAXA_MAXIMA * fracao / 0.03                                     # aquecimento
        else:
            resto = (fracao - 0.03) / 0.97
            taxa = TAXA_MAXIMA * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * resto)))  # resfriamento
        self.model.optimizer.learning_rate.assign(taxa)

    def on_epoch_end(self, conferencia, logs=None):
        h = self.historico
        h["passo"].append(int(self.model.optimizer.iterations.numpy()))
        h["minuto"].append((time.time() - self.inicio) / 60)
        h["treino"].append(logs["loss"])
        h["validacao"].append(logs["val_loss"])
        frase = " ".join(gerar(self.model, "A casa de", 25).split())
        print(f"{h['minuto'][-1]:4.1f} min  passo {h['passo'][-1]:>6}  treino {logs['loss']:.2f}  "
              f"validação {logs['val_loss']:.2f}  | {frase}")


relogio = Relogio(MINUTOS_DE_TREINO)
melhor = keras.callbacks.ModelCheckpoint("meu_llm_gpu.weights.h5", monitor="val_loss",
                                         save_best_only=True, save_weights_only=True)

## 8. O treino

O `fit` repete o ciclo: sorteia um lote, prevê, mede a perda, calcula o
gradiente e corrige os 809.728 pesos. O primeiro passo demora um pouco
mais, porque o XLA compila o modelo. Depois, cada linha é uma conferência.

In [ ]:
modelo.fit(lotes(treino), steps_per_epoch=PASSOS_POR_CONFERENCIA, epochs=10_000,
           validation_data=lotes(validacao), validation_steps=20,
           callbacks=[relogio, melhor], verbose=0)
h = relogio.historico
print(f"\nFim: {br(h['passo'][-1])} passos, {h['passo'][-1] * LOTE * BLOCO / 1e6:.0f} milhões de tokens lidos, "
      f"melhor validação {min(h['validacao']):.2f} (perplexidade {math.exp(min(h['validacao'])):.0f})")

## 9. A curva

A perda despenca no começo e depois quase para. Quando a validação para
de cair e o treino continua, o modelo começou a decorar.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(h["minuto"], h["treino"], marker="o", label="treino")
plt.plot(h["minuto"], h["validacao"], marker="o", label="validação")
plt.axhline(math.log(1024), color="gray", linestyle=":", label="chute ao acaso")
plt.xlabel("minutos de treino")
plt.ylabel("perda")
plt.legend()
plt.show()

## 10. O modelo escrevendo

Carregamos o melhor modelo (o da menor validação) e geramos texto com
três temperaturas: baixa repete, alta inventa palavras.

In [ ]:
modelo.load_weights("meu_llm_gpu.weights.h5")
for temperatura in [0.3, 0.8, 1.3]:
    print(f"--- temperatura {temperatura}")
    print(gerar(modelo, "Não sei se a senhora", 60, temperatura), "\n")

## 11. Leve o modelo para casa

Os pesos têm os mesmos nomes do modelo das Aulas 10 e 11: o arquivo
carrega em qualquer `montar_modelo()` do curso com `load_weights`.

In [ ]:
try:
    from google.colab import files
    files.download("meu_llm_gpu.weights.h5")
except ImportError:
    print("Fora do Colab: o arquivo meu_llm_gpu.weights.h5 está na pasta deste notebook.")